In [1]:
import pandas as pd
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

def generate_chat_history_summary(chat_history: str) -> str:
    """
    Generate a summary of the chat history.
    """

    system_prompt = '''
    You are an expert customer service representative.
    Your task is to summarize the chat history into a short paragraph, with a focus on customer's tone, sentiment, and all speaker's action.
    '''

    config = genai.types.GenerateContentConfig(
        system_instruction = system_prompt,
        response_mime_type="text/plain",
        temperature=0.0,
    )

    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = chat_history,
        config = config,
    )

    return response.text

def get_history_data(history_df, convo_id, round):
    """
    Get the history data for a given conversation and round.
    Output: chat history: str
    chat history should include all the text from the beginning of the conversation to the previous round, plus the current round Customer's text.
    In the format of "Speaker [Speaker Action] : Text"
    """

    
    history_slice = history_df[(history_df['round'] <= round) & (history_df['convo_id'] == convo_id)].copy()
    prev_rounds = history_slice[history_slice['round'] < round].copy()
    current_round = history_slice[(history_slice['round'] == round) & (history_slice['speaker'] == "Customer")].copy()

    rows = []
    for _, row in prev_rounds.iterrows():
        rows.append(f"{row['speaker']} [{row['action']}] : {row['text']}")
    for _, row in current_round.iterrows():
        rows.append(f"{row['speaker']} [{row['action']}] : {row['text']}")
    chat_history = "\n".join(rows)
    return generate_chat_history_summary(chat_history)

agent_action_map = {
    "A_Persuade": 1,
    "A_Incentive": 2,
    "B_Persuade": 3,
    "B_Incentive": 4,
    "C_Persuade": 5,
    "C_Incentive": 6,
    "Closing": 7}

def generate_transition_df(history_df):

    transition_prev = history_df[(history_df['round'] > 2) & (history_df['speaker'] == "Agent")].copy()
    transition_prev['chat_history'] = transition_prev.apply(
        lambda row: get_history_data(history_df, row['convo_id'], row['round'] ), axis = 1)
    transition_prev['action'] = transition_prev['action'].map(agent_action_map)

    transition_prev_copy = transition_prev.copy()
    transition_prev_copy['round'] = transition_prev_copy['round'] - 1
    transition_prev_copy = transition_prev_copy[['convo_id', 'round', 'chat_history']]
    transition_prev_copy.rename(columns = {'chat_history': 'chat_history_1'}, inplace = True)
    transition_prev.rename(columns = {'chat_history': 'chat_history_0'}, inplace = True)

    transition_df = pd.merge(transition_prev, transition_prev_copy, on = ['convo_id', 'round'], how = 'left')

    return transition_df

transition_df = pd.DataFrame()

In [2]:
history_batch_1 = pd.read_csv("all_histories_20260129_124255.csv")
final_state_batch_1 = pd.read_csv("all_final_states_20260129_124255.csv")
transition_batch_1 = generate_transition_df(history_batch_1)[['convo_id', 'round', 'chat_history_0', 'chat_history_1', 'action', 'end_convo']]

final_state_copy = final_state_batch_1[['convo_id', 'final_state']].copy()
final_state_copy['end_convo'] = True
transition_batch_1 = pd.merge(transition_batch_1, final_state_copy, on = ['convo_id', 'end_convo'], how = 'left')
transition_batch_1['chat_history_1'] = transition_batch_1['chat_history_1'].fillna(transition_batch_1['final_state'])

transition_df = transition_batch_1.copy()
transition_df.to_csv("transition_df.csv", index = False)
transition_df.head()

,convo_id,round,chat_history_0,chat_history_1,action,end_convo,final_state
0,0,3,"The customer, initially direct and slightly sk...",The chat begins with the agent greeting the cu...,6,False,NaN
1,0,4,The chat begins with the agent greeting the cu...,The customer initiated the chat seeking genuin...,2,False,NaN
2,0,5,The customer initiated the chat seeking genuin...,Customer buys,4,True,Customer buys
3,1,3,The customer began with a somewhat demanding a...,Customer leaves call,1,True,Customer leaves call
4,2,3,The agent greeted the customer and offered ass...,"The customer, initially seeking durable and wa...",5,False,NaN


In [4]:
history_batch_2 = pd.read_csv(r"all_histories_20260129_132224.csv")
final_state_batch_2 = pd.read_csv(r"all_final_states_20260129_132224.csv")

history_batch_2['convo_id'] += 110
final_state_batch_2['convo_id'] += 110

transition_batch_2 = generate_transition_df(history_batch_2)[['convo_id', 'round', 'chat_history_0', 'chat_history_1', 'action', 'end_convo']]
final_state_batch_2 = final_state_batch_2[['convo_id', 'final_state']]
final_state_batch_2['end_convo'] = True
transition_batch_2 = pd.merge(transition_batch_2, final_state_batch_2, on = ['convo_id', 'end_convo'], how = 'left')
transition_batch_2['chat_history_1'] = transition_batch_2['chat_history_1'].fillna(transition_batch_2['final_state'])

transition_df = pd.concat([transition_df, transition_batch_2], ignore_index = True)
transition_df.to_csv("transition_df.csv", index = False)

In [5]:
history_batch_3 = pd.read_csv(r"all_histories_20260129_135836.csv")
final_state_batch_3 = pd.read_csv(r"all_final_states_20260129_135836.csv")

history_batch_3['convo_id'] += 220
final_state_batch_3['convo_id'] += 220

transition_batch_3 = generate_transition_df(history_batch_3)[['convo_id', 'round', 'chat_history_0', 'chat_history_1', 'action', 'end_convo']]
final_state_batch_3 = final_state_batch_3[['convo_id', 'final_state']]
final_state_batch_3['end_convo'] = True
transition_batch_3 = pd.merge(transition_batch_3, final_state_batch_3, on = ['convo_id', 'end_convo'], how = 'left')
transition_batch_3['chat_history_1'] = transition_batch_3['chat_history_1'].fillna(transition_batch_3['final_state'])

transition_df = pd.concat([transition_df, transition_batch_3], ignore_index = True)
transition_df.to_csv("transition_df.csv", index = False)

In [6]:
history_batch_4 = pd.read_csv(r"all_histories_20260129_142917.csv")
final_state_batch_4 = pd.read_csv(r"all_final_states_20260129_142917.csv")

history_batch_4['convo_id'] += 330
final_state_batch_4['convo_id'] += 330

transition_batch_4 = generate_transition_df(history_batch_4)[['convo_id', 'round', 'chat_history_0', 'chat_history_1', 'action', 'end_convo']]
final_state_batch_4 = final_state_batch_4[['convo_id', 'final_state']]
final_state_batch_4['end_convo'] = True
transition_batch_4 = pd.merge(transition_batch_4, final_state_batch_4, on = ['convo_id', 'end_convo'], how = 'left')
transition_batch_4['chat_history_1'] = transition_batch_4['chat_history_1'].fillna(transition_batch_4['final_state'])

transition_df = pd.concat([transition_df, transition_batch_4], ignore_index = True)
transition_df.to_csv("transition_df.csv", index = False)